# Solar PV Generation — Exploratory Data Analysis

Dataset: NREL Washington State solar PV actuals (2006), 5-min resolution.  
Target: `Power(MW)` — continuous regression target.

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Overview

In [ ]:
from data.loader import load_all

# Load Actual 5-min readings; subsample to 500k rows for speed
df_full = load_all(prefix='Actual')
df = df_full.sample(n=min(500_000, len(df_full)), random_state=42).copy()

# Derived time features used throughout
df['hour']      = df['LocalTime'].dt.hour
df['month']     = df['LocalTime'].dt.month
df['capacity_mw'] = df['capacity'].str.replace('MW', '').astype(float)

print(f'Full dataset : {len(df_full):,} rows')
print(f'Working sample: {len(df):,} rows')
print(f'\nShape: {df.shape}')
print(f'\nColumns & dtypes:')
print(df.dtypes.to_string())
df.head()

## 2. Target Analysis

In [ ]:
target = 'Power(MW)'

print('Target summary statistics:')
print(df[target].describe().round(3).to_string())
print(f'\nSkewness : {df[target].skew():.3f}')
print(f'Kurtosis : {df[target].kurt():.3f}')
print(f'Zero values: {(df[target] == 0).sum():,} ({(df[target] == 0).mean()*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(df[target], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Power(MW) — Full Distribution', fontsize=13)
axes[0].set_xlabel('Power (MW)')
axes[0].set_ylabel('Count')

# Non-zero only (daylight hours)
nz = df[df[target] > 0][target]
axes[1].hist(nz, bins=80, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].set_title('Power(MW) — Non-Zero Only (Daylight)', fontsize=13)
axes[1].set_xlabel('Power (MW)')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable Distribution', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
null_counts  = df.isnull().sum()
null_pct     = (df.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame({'count': null_counts, 'pct': null_pct})
print('Missing value summary:')
print(null_summary.to_string())

# Build a column-level null rate bar chart (heatmap needs 2-D data;
# with only column-level nulls a bar chart is clearer)
fig, ax = plt.subplots(figsize=(12, 5))
null_matrix = df.isnull().astype(int)
sample_null = null_matrix.sample(n=min(5000, len(null_matrix)), random_state=42)
sns.heatmap(
    sample_null.T,
    ax=ax,
    cbar=True,
    cmap='YlOrRd',
    yticklabels=df.columns,
    xticklabels=False,
    linewidths=0
)
ax.set_title('Missing Value Heatmap (sample of 5,000 rows)', fontsize=13)
ax.set_xlabel('Row index (sampled)')
ax.set_ylabel('Column')
plt.tight_layout()
plt.show()

## 4. Feature Distributions

In [ ]:
# 9 features for the 3x3 grid
plot_features = [
    ('Power(MW)',    'steelblue'),
    ('lat',         'teal'),
    ('lon',         'teal'),
    ('capacity_mw', 'slateblue'),
    ('hour',        'darkorange'),
    ('month',       'darkorange'),
    ('Power(MW)',   'crimson'),   # DPV slice
    ('Power(MW)',   'forestgreen'),  # UPV slice
    ('capacity_mw', 'goldenrod'),
]

labels = [
    'Power(MW)', 'Latitude', 'Longitude',
    'Site Capacity (MW)', 'Hour of Day', 'Month',
    'Power(MW) — DPV only', 'Power(MW) — UPV only',
    'Capacity (MW) distribution'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, (ax, (col, color), label) in enumerate(zip(axes, plot_features, labels)):
    if 'DPV' in label:
        data = df[df['pv_type'] == 'DPV'][col]
    elif 'UPV' in label:
        data = df[df['pv_type'] == 'UPV'][col]
    else:
        data = df[col]
    ax.hist(data.dropna(), bins=40, color=color, edgecolor='white', linewidth=0.3)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=15)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr_cols = ['Power(MW)', 'lat', 'lon', 'capacity_mw', 'hour', 'month']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle only
sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Features vs Target

In [ ]:
# Regression: scatter plots of 3 strongest correlates with target
corr_with_target = corr['Power(MW)'].drop('Power(MW)').abs().sort_values(ascending=False)
top3 = corr_with_target.head(3).index.tolist()
print(f'Top 3 correlates with Power(MW): {top3}')

# Downsample to 20k points for readable scatter plots
sample = df.sample(n=min(20_000, len(df)), random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

for ax, feat, color in zip(axes, top3, colors):
    ax.scatter(sample[feat], sample['Power(MW)'], alpha=0.15, s=4, color=color)
    # Trend line
    m, b = np.polyfit(sample[feat].fillna(0), sample['Power(MW)'], 1)
    x_range = np.linspace(sample[feat].min(), sample[feat].max(), 100)
    ax.plot(x_range, m * x_range + b, color='black', linewidth=1.5, linestyle='--', label='trend')
    r = df[['Power(MW)', feat]].corr().iloc[0, 1]
    ax.set_title(f'Power(MW) vs {feat}\n(r = {r:.2f})', fontsize=11)
    ax.set_xlabel(feat)
    ax.set_ylabel('Power (MW)')
    ax.legend(fontsize=8)

plt.suptitle('Top Feature Relationships with Target (20k sample)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Zero inflation**: roughly 70% of all readings are 0 MW — solar panels produce nothing at night, so any model must account for this bimodal distribution (zero vs positive output).
- **Capacity drives output**: site capacity (MW nameplate) shows the strongest correlation with actual power output; UPV (utility-scale) sites generate significantly more than DPV (distributed) sites and exhibit higher variance.
- **Strong hourly pattern**: power peaks mid-day (hours 11–14) and drops to zero outside daylight hours; hour-of-day is the most important temporal feature for prediction.
- **Mild geographic signal**: latitude shows a weak negative correlation with output — sites further north (higher lat) in Washington State tend to generate slightly less power, consistent with lower solar irradiance.
- **No missing data**: the dataset is complete with 0% nulls across all columns, so imputation is not required; the main data quality concern is the constant `year` column (2006 only), which carries no predictive information.